# 🎵 تدريب صوت العميل (أقصى تشابه — Fine-tune)

هنا **بتدرّب الموديل على صوت العميل نفسه** (بعد موافقته المكتوبة) عشان الشبه والنغمة
يطلعوا بأعلى درجة — مش مجرد استنساخ سريع.

1. `Runtime` → `Change runtime type` → **T4 GPU** → `Save`
2. شغّل الخلايا **بالترتيب** (`Shift + Enter`)
3. سجّلات العميل → الداتاسِت → التدريب → **zips للتنزيل**
4. (اختياري) تشغيل سيرفر الموديل بصوت المدرب → **🔗 رابط + 🔑 مفتاح** للتطبيق

> ⏱️ التدريب بياخد **~30–60 دقيقة** على T4 المجاني حسب كمية التسجيلات.
> 🕒 ممكن تسيب الجلسة شغّالة وتنزل بعد ما يخلص.

In [ ]:
# 1) نتأكد إن GPU شغّال
import torch
print("CUDA متاح؟", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ من غير GPU! ارجع: Runtime → Change runtime type → T4 GPU")
else:
    print("✅ GPU جاهز:", torch.cuda.get_device_name(0))

In [ ]:
# 2) تثبيت المكتبات (دقيقتين تقريبًا)
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q peft accelerate
!pip install -q voicetut-tts soundfile pyloudnorm scipy numpy google-cloud-storage
print("✅ المكتبات اتثبّتت")

In [ ]:
# 3) هات الكود (أول مرة ينسخ، وبعدها بيعمل تحديث تلقائي)
%cd /content
import os

REPO = "https://github.com/pharmacygamila-ship-it/egyptian-voice-studio.git"
if os.path.isdir("project/.git"):
    %cd project
    !git pull -q
    print("✅ اتحدّث لآخر نسخة")
else:
    !git clone -q {REPO} project
    %cd project
    print("✅ اتنسخ لأول مرة")

## 4) ارفع تسجيلات العميل 📁

جهّز **ملف مضغوط zip** فيه تسجيلات العميل، كل مقطع = ملف + نصه. أفضل طريقة:

- 10–30 دقيقة كلام **نضيف** (أوضة هادية، من غير موسيقى/صدى — نفس أسلوب الإعلان)
- ملف لكل جملة: `clip01.wav` + `clip01.txt` (جنبه بنفس الاسم فيه النص)
  - **أو** ملف واحد اسمه `transcripts.txt` بصيغة:  `clip01.wav|النص اللي بيتقال`
- بصغى wav أو mp3 — **متوسط 3–15 ثانية** للمقطع الواحد

In [ ]:
# شغّل الـ zip ده من الجهاز
from google.colab import files
up = files.upload()          # ← اضغط Browse واختار الزومبي
zip_name = next(iter(up))

# فك الضغط في مجلد voice_raw
import zipfile, os
os.makedirs("/content/voice_raw", exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/voice_raw")

raw = [f for f in os.listdir("/content/voice_raw")
       if f.lower().endswith((".wav", ".mp3", ".flac", ".m4a", ".ogg"))]
print("✅ عدد المقاطع:", len(raw))
for f in sorted(raw)[:8]:
    print("   -", f)
if not raw:
    raise SystemExit("⚠️ مفيش ملفات صوت في الزومبي — تأكد إن الملفات جوه مجلد/مباشرة")

## 5) تجهيز الداتاسِت

بيحوّل التسجيلات لـ**مقاطع منظّفة 24kHz** + يقسّمها تدريب/تحقق + تقرير جودة.
لو فيه مقاطع مرفوضة (مثلاً قصيرة جدًا أو من غير نص)، هيظهر سببها في التقرير.

In [ ]:
# تجهيز الداتاسِت (نص دقيقة)
!python finetune/prepare_dataset.py --in /content/voice_raw --out finetune/dataset
print("----")
import json
rep = json.load(open("finetune/dataset/report.json", encoding="utf-8"))
print("إجمالي:", rep.get("total_seconds"), "ثانية | تدريب:", rep.get("train"), "| تحقق:", rep.get("val"), "| مرفوض:", rep.get("dropped"))

## 6) فحص الداتاسِت وخطة التدريب (قبل ما نضيع وقت على GPU)

In [ ]:
# فحص + خطة تقريبية
!python finetune/train_lora.py --check

## 7) 🎬 التدريب

- لو التسجيلات **أقل من 5 دقائق**: التدريب مش هيضيف كتير — الأفضل استخدم **استنساخ** بـ Zero-shot مباشرة.
- **الاسم**: حروف إنجليزي بس (هو اللي هيظهر في أسماء الملفات).

In [ ]:
# ⚠️ عدّل الاسم هنا لو حابب، وبعدين شغّل الخلية دي (التدريب نفسه ~30-60 دقيقة)
NAME = "voice_client"
!python finetune/train_lora.py --dataset finetune/dataset --name {NAME}

print("✅ التدريب خلص")
import json, os
card = json.load(open(f"finetune/runs/{NAME}/{NAME}.json", encoding="utf-8"))
for k, v in card.items():
    print("  ", k, ":", v)

## 8) 📥 نزّل نتيجة التدريب

هينزّل 3 ملفات:
- `<NAME>_adapter.zip` — الـ adapter الصغير (لو عايز تحمّله فوق الموديل الأساسي)
- `<NAME>_model.zip` — **الموديل الكامل المدموج** (ده الأهم — يحمّله السيرفر)
- بطاقة `<NAME>.json` — بيانات الصوت

> 🛟 احتفظ بـ `_model.zip` في مكان آمن — ده **صوت العميل المدرب** وبدونه مفيش حاجة.

In [ ]:
# تنزيل الملفات
import os
from google.colab import files

card_dir = f"/content/project/finetune/runs/{NAME}"
for f in sorted(os.listdir(card_dir)):
    p = os.path.join(card_dir, f)
    if f.lower().endswith(".zip"):
        print("⬇️ بتنزيل:", f, "(", round(os.path.getsize(p)/1e6, 1), "MB )")
        files.download(p)

## (اختياري) 🚀 شغّل سيرفر الموديل **بصوت العميل المدرب**

طيّب عشان **التطبيق عندك** يولّد بصوت المدرب: هتشغّل سيرفر الموديل هنا، وهو حامل
**الموديل المدموج** (`TTS_MODEL`). التطبيق يتصل بيه بنفس الطريقة (**المفتاح**).

> 💡 أحسن نتيجة: في التطبيق استخدم تبويب **🧬 استنساخ** ومقطع **10–15 ثانية** من العميل
> كمرجع — الموديل المدرب بيخلّي الشبه أعلى بكتير من غيره.

In [ ]:
# تشغيل السيرفر بالموديل المدرب + النفق العام + المفتاح
import os, sys, re, time, secrets, subprocess, pathlib, requests

os.environ["TTS_ENGINE"] = "local"
os.environ["TTS_DATA_DIR"] = "/content/project/data"
os.environ["TTS_OUTPUT_DIR"] = "/content/project/data/outputs"
os.environ["TTS_MODEL"] = f"/content/project/finetune/runs/{NAME}/merged"   # 🎯 الموديل المدرب
TOKEN = "vs_" + secrets.token_urlsafe(24)
os.environ["TTS_SERVER_TOKEN"] = TOKEN

subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/project",
    stdout=open("/content/server.log", "a"), stderr=subprocess.STDOUT,
)
up = False
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:8000/health", timeout=2); up = True; break
    except Exception:
        time.sleep(1)
print("✅ السيرفر شغّال" if up else "⚠️ السيرفر مش راضي يقوم — بص /content/server.log")

pathlib.Path("/content/tunnel.log").write_text("")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=open("/content/tunnel.log", "a"), stderr=subprocess.STDOUT,
)
URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                  pathlib.Path("/content/tunnel.log").read_text(errors="ignore"))
    if m:
        URL = m.group(0); break

print("\n" + "=" * 66)
print("🔗 رابط السيرفر :", URL or "⚠️ مش ظهر — بص /content/tunnel.log")
print("🔑 المفتاح       :", TOKEN)
print("ميدل المدرب       :", os.environ["TTS_MODEL"])
print("=" * 66)
print("\nافتح التطبيق → ⚙️ الإعدادات → server → الصق الرابط والمفتاح → تطبيق وتشغيل")

## 📌 ملاحظات

- **القانوني (إلزامي):** موافقة **مكتوبة** من صاحب الصوت قبل أي استنساخ/تدريب،
  وممنوع انتحال أو تضليل. احتفظ بالتسجيلات بأمان.
- التدريب ده بيحسّن **أداء ونغمة الصوت** لأي جملة — وللنطق الصحيح لكل كلمة،
  التطبيق بيعمل **تطبيع مصري + تشكيل** والـ **فاحص نطق** يتحقق بالحرف.
- كولاب المجاني بيقفل الجلسة بعد مدة — نزّل الـ zips **مباشرة** بعد التدريب.
- عايز سيرفر دائم 24 ساعة وتحكم كامل؟ استخدم `deploy/modal_app.py` على Modal/RunPod.